In [ ]:
import json

import pandas as pd
import great_expectations as gx
import synapseclient

from agoradatatools.gx import GreatExpectationsRunner

context = gx.get_context(project_root_dir='../src/agoradatatools/great_expectations')

from expectations.expect_column_values_to_have_list_members_of_type import ExpectColumnValuesToHaveListMembersOfType

# Create Expectation Suite for Drug Info Data

## Get Example Data File

In [ ]:
syn = synapseclient.Synapse()
syn.login()

In [ ]:
drug_info_data_file = syn.get("syn73880976").path

## Create Validator Object on Data File

In [ ]:
df = pd.read_json(drug_info_data_file)
nested_columns = ['linked_targets', 'drug_nominations']
df = GreatExpectationsRunner.convert_nested_columns_to_json(df, nested_columns)
validator = context.sources.pandas_default.read_dataframe(df)
validator.expectation_suite_name = "drug_info"

## Add Expectations to Validator Object For Each Column

In [ ]:
# common_name
validator.expect_column_values_to_be_of_type("common_name", "str")
validator.expect_column_values_to_not_be_null("common_name")
validator.expect_column_value_lengths_to_be_between("common_name", min_value=1, max_value=250)

In [ ]:
# description (nullable; sourced from OpenTargets metadata)
validator.expect_column_values_to_be_of_type("description", "str")
validator.expect_column_value_lengths_to_be_between("description", min_value=1, max_value=1000)

In [ ]:
# iupac_id (nullable)
validator.expect_column_values_to_be_of_type("iupac_id", "str")
validator.expect_column_value_lengths_to_be_between("iupac_id", min_value=1, max_value=1000)

In [ ]:
# chembl_id
validator.expect_column_values_to_be_of_type("chembl_id", "str")
validator.expect_column_values_to_not_be_null("chembl_id")
# chembl_id is not unique: combination drugs can share a ChEMBL ID across rows
validator.expect_column_values_to_match_regex("chembl_id", "^CHEMBL\\d+$")

In [ ]:
# drug_bank_id (nullable; sourced from OpenTargets metadata)
validator.expect_column_values_to_be_of_type("drug_bank_id", "str")
validator.expect_column_values_to_match_regex(
    "drug_bank_id",
    regex="^DB\\d+$",
    row_condition="drug_bank_id.notnull()",
    condition_parser="pandas",
)

In [ ]:
# modality (nullable; populated from OpenTargets metadata)
validator.expect_column_values_to_be_of_type("modality", "str")
validator.expect_column_values_to_be_in_set("modality", ["Small molecule", "Protein"])

In [ ]:
# year_of_first_approval (nullable; loaded as float because of nulls)
validator.expect_column_values_to_be_of_type("year_of_first_approval", "float")
validator.expect_column_values_to_be_between("year_of_first_approval", min_value=1938, max_value=2050)

In [ ]:
# maximum_clinical_trial_phase (nullable)
validator.expect_column_values_to_be_of_type("maximum_clinical_trial_phase", "str")
validator.expect_column_values_to_be_in_set(
    "maximum_clinical_trial_phase",
    ["Phase I", "Phase II", "Phase III", "Phase IV", "Preclinical", "Unknown"],
)

In [ ]:
# aliases (list of strings; may be empty)
validator.expect_column_values_to_be_of_type("aliases", "list")
validator.expect_column_values_to_not_be_null("aliases")
validator.expect_column_values_to_have_list_members_of_type(column="aliases", member_type="str")

In [ ]:
# mechanisms_of_action (list of strings; may be empty)
validator.expect_column_values_to_be_of_type("mechanisms_of_action", "list")
validator.expect_column_values_to_not_be_null("mechanisms_of_action")
validator.expect_column_values_to_have_list_members_of_type(column="mechanisms_of_action", member_type="str")

In [ ]:
# linked_targets (nested array of gene objects)
validator.expect_column_values_to_be_of_type("linked_targets", "str")
validator.expect_column_values_to_not_be_null("linked_targets")
with open("../src/agoradatatools/great_expectations/gx/json_schemas/drug_info/linked_targets.json", "r") as file:
    linked_targets_schema = json.load(file)
validator.expect_column_values_to_match_json_schema("linked_targets", json_schema=linked_targets_schema)

In [ ]:
# drug_nominations (nested array of nomination objects)
validator.expect_column_values_to_be_of_type("drug_nominations", "str")
validator.expect_column_values_to_not_be_null("drug_nominations")
with open("../src/agoradatatools/great_expectations/gx/json_schemas/drug_info/drug_nominations.json", "r") as file:
    drug_nominations_schema = json.load(file)
validator.expect_column_values_to_match_json_schema("drug_nominations", json_schema=drug_nominations_schema)

## Save Expectation Suite

In [ ]:
validator.save_expectation_suite(discard_failed_expectations=False)

## Create Checkpoint and View Results

In [ ]:
checkpoint = context.add_or_update_checkpoint(
    name="agora-test-checkpoint",
    validator=validator,
)
checkpoint_result = checkpoint.run()
context.view_validation_result(checkpoint_result)

## Build Data Docs - Click on Expectation Suite to View All Expectations

In [ ]:
context.build_data_docs()
context.open_data_docs()